<a href="https://colab.research.google.com/github/mobarakol/tutorial_notebooks/blob/main/mamba_ssm_scratch_hugginface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Scratch

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MinimalSSM(nn.Module):
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # Learnable parameters for the SSM (A, D)
        self.A_log = nn.Parameter(torch.randn(d_model, d_state))
        self.D = nn.Parameter(torch.randn(d_model))

        # --- THE FIX IS HERE ---
        # We need to project x into:
        # 1. delta (size: d_model)
        # 2. B     (size: d_state)
        # 3. C     (size: d_state)
        # Total output size = d_model + d_state + d_state
        self.x_proj = nn.Linear(d_model, d_model + d_state * 2)

        # Output projection
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        """
        x: (Batch, Seq_Len, d_model) -> (B, 8, 512)
        """
        batch, seq_len, dim = x.shape

        # Initialize hidden state (h) to zeros
        h = torch.zeros(batch, dim, self.d_state, device=x.device)

        outputs = []

        for t in range(seq_len):
            x_t = x[:, t, :] # (Batch, Dim)

            # 1. Project input to get dynamic parameters
            projected = self.x_proj(x_t)

            # Split into delta, B, C
            # delta: (Batch, Dim)
            # B:     (Batch, State)
            # C:     (Batch, State)
            delta, B, C = torch.split(projected, [dim, self.d_state, self.d_state], dim=-1)

            # Softplus to ensure positive step size delta
            delta = F.softplus(delta)

            # A calculation (simplified)
            A = -torch.exp(self.A_log)

            # Discretize A and B
            # A_bar: (Batch, Dim, State)
            # B_bar: (Batch, Dim, State)
            A_bar = torch.exp(delta.unsqueeze(-1) * A)
            B_bar = delta.unsqueeze(-1) * B.unsqueeze(1)

            # 2. Update State: h_t = A_bar * h_{t-1} + B_bar * x_t
            # x_t needs to be broadcast to (Batch, Dim, 1) to multiply B_bar
            h = A_bar * h + B_bar * x_t.unsqueeze(-1)

            # 3. Output: y_t = C * h_t + D * x_t
            # C needs to be broadcast to (Batch, 1, State)
            y_t = (h * C.unsqueeze(1)).sum(dim=-1) + self.D * x_t

            outputs.append(y_t)

        out = torch.stack(outputs, dim=1)
        return self.out_proj(out)

# --- Usage ---
input_features = torch.randn(2, 8, 512)
model = MinimalSSM(d_model=512)
output = model(input_features)

print(f"Minimal PyTorch Output Shape: {output.shape}")
# Output: torch.Size([2, 8, 512])

Minimal PyTorch Output Shape: torch.Size([2, 8, 512])


Huggingface:

In [4]:
import torch
from transformers import MambaConfig, MambaModel

# --- CORRECT CONFIGURATION ---
config = MambaConfig(
    hidden_size=512,      # Changed from d_model to hidden_size
    num_hidden_layers=1,  # Keep it small for feature extraction
    use_cache=False       # Not needed for non-generative tasks
)

# Initialize model
model = MambaModel(config)

# Input: (Batch=2, Seq_Len=8, Features=512)
input_features = torch.randn(2, 8, 512)

# Pass through model
output = model(inputs_embeds=input_features).last_hidden_state

print(f"Output shape: {output.shape}")
# Output: torch.Size([2, 8, 512])

Output shape: torch.Size([2, 8, 512])
